# Calibrate using TensorFlow

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd

import fates_calibration_library.emulator_functions as em
import fates_calibration_library.utils as utils
from fates_calibration_library.TFClass import TFEmulator
import tensorflow as tf

import importlib

2025-07-02 10:20:23.211468: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 10:20:24.333341: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 10:20:25.920232: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 10:20:25.921755: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-02 10:20:40.316681: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

## Set Up
Load files, set up ensemble information

In [ ]:
# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'
fig_dir = '/glade/work/afoster/FATES_calibration/figures'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# default parameter file
default_param = xr.open_dataset(os.path.join(param_dir, 
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]

# normalized values for parameters
default_norm = pd.read_csv(os.path.join(param_dir, 'normalized_parameters.csv'), index_col=[0])

# variables to calibrate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

# information about variables
obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

# PFT ids
pft_id_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_pft_ids.yaml'
pft_ids = utils.get_config_file(pft_id_config)

In [ ]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13, 14],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            }
           }

In [ ]:
# choose ensemble
ensemble = 'dompft'

### Load Latin Hypercube Key

In [ ]:
lhc_key = pd.read_csv(ens_dict[ensemble]['lhc_key_file'], index_col=[0])
lhc_key = lhc_key.drop(columns=['ensemble'])
param_names = lhc_key.columns
num_params = len(param_names)

### Load Observations

In [ ]:
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

### Load Parameter Sensitivity

In [ ]:
sens_df = pd.read_csv(os.path.join(emulator_dir, f'sensitivity_df_{ensemble}.csv'), index_col=[0])

## Calibration

In [ ]:
pft = 1
pft_name = all_pfts[pft-1]
pft_id = pft_ids[pft_name]

In [ ]:
# get observations for this pft
obs_pft = obs[obs.pft == pft_name]

# get sensitivity data for this pft
sens_pft = sens_df[sens_df.pft == pft_id]

# sum up sobol indices
# sobol_indices = tf.constant([sens_pft[sens_pft.parameter == p]['ST'].sum() for p in param_names], dtype=tf.float64)

# get default values for this pft
default_pft = default_norm[default_norm.pft == pft]
default_pft = default_pft.drop(columns=['pft'])

# convert to tf object
# X_default_all = tf.constant(default_pft.to_numpy(), dtype=tf.float64)

In [ ]:
# # get parameter sensitivity information
# sobol_threshold = 0.01
# optimize_mask = sobol_indices > sobol_threshold
# fixed_indices = tf.where(tf.logical_not(optimize_mask))[:, 0]
# X_fixed = tf.gather(X_default_all, fixed_indices, axis=1)

# opt_indices = tf.where(optimize_mask)[:, 0]
# num_optimized = tf.reduce_sum(tf.cast(optimize_mask, tf.int32))
# X_default_opt = tf.constant(tf.gather(X_default_all, opt_indices, axis=1), dtype=tf.float64)

In [ ]:
# stack targets, sds, and emulators for all variables
targets = []
sds = []
emulators = []
for variable in calibration_vars:
    
    # observations for this pft and variable
    obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])
    
    # convert to tf objects
    targets.append(tf.convert_to_tensor(obs_mean, dtype=tf.float64))
    sds.append(tf.convert_to_tensor(obs_sd, dtype=tf.float64))

    # load the emulator
    emulators.append(TFEmulator(emulator_dir, pft=pft_id, variable=variable))

In [ ]:
config = {
    'checkpoint_dir': '/glade/work/afoster/FATES_calibration/checkpoints',
    'learning_rate': 1e-3,
    'lr_decay_steps': 300,
    'maxiter': 3000,
    'checkpoint_n': 10,
    'epsilon': 0.5,
    'lambda_penalty': None,
    'barrier_strength': 0,
    'earlystop_pct': 90.0,
    'loss_fn': em.implausibility_loss,
    'default_penalty_fn': em.default_penalty_l1,
    'barrier_penalty_fn': em.barrier_penalty,
}

In [ ]:
X_opt_batch = tf.Variable(
    tf.random.uniform(shape=(1000, num_optimized),
                      minval=0.0,
                      maxval=1.0,
                      dtype=tf.float64),
    dtype=tf.float64, trainable=True, name='X'
)

In [ ]:
importlib.reload(em)

In [ ]:
batch_size = 100
n_opt = 14

In [ ]:
tensor = tf.zeros([batch_size, n_opt], dtype=tf.dtypes.float64)
new_cols = tf.tile(tf.reshape(X_fixed, (1, -1)), [batch_size, 1])
indices = fixed_indices.numpy()
for i, insert_idx in enumerate(indices):
    
    slice = tf.slice(new_cols, begin=[0, i], size=[batch_size, 1])
    
    if insert_idx == 0:
        left = slice
        right = tensor
        result = tf.concat([left, right], axis=1)
    else:
        left = tensor[:, :insert_idx]
        right = tensor[:, insert_idx:]
        result = tf.concat([left, slice, right], axis=1)
    tensor = result

In [ ]:
X_opt, logs = em.run_optimization(X_init, X_fixed, fixed_indices.numpy(), emulators, targets, sds, X_default_opt, config)

In [ ]:
np.argmin(logs['losses'][2999])

In [ ]:
X_opt[930]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(logs['total_loss'], label='Total loss')
plt.plot(logs['data_loss'], label='Data loss')
plt.plot(logs['default_penalty'], label='Penalty loss')
plt.plot(logs['barrier_penalty'], label='Barrier loss')
plt.xlabel('Step')
plt.ylabel('Loss value')
plt.legend()
plt.grid(True)
plt.title('Loss components over optimization steps')
#plt.ylim(0,5)
#plt.savefig('loss_plot_unifRandom_minError_clipNoPenalties.png')

In [ ]:
calibrated_params = pd.DataFrame(X_opt, columns=param_names)

In [ ]:
# Initializing the input
indices = [[0, 1, 5], [2, 4, 3, 6]]
data = [[1, 2, 3], [4, 5, 6, 7]]

# Printing the input
print('indices:', indices)
print('data: ', data)

# Calculating result
x = tf.dynamic_stitch(indices, data)

# Printing the result
print('x: ', x)